In [1]:
import os, sys, numpy as np, pandas as pd
sys.path.insert(0, os.getcwd())
from sklearn.model_selection import KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.preprocessing import TargetEncoder
from pipeline import load_data, build_pipeline, build_preprocessor, RAW_COLS

X, y = load_data(os.path.join(path, "Melbourne_housing_FULL.csv"))
cv = KFold(5, shuffle=True, random_state=42)

def leaky_target_encoding():
    """Same pipeline, but Suburb is target-encoded OUTSIDE it, fitted on the full data."""
    pre = build_preprocessor("target")
    pre.set_params(encode_scale__suburb="drop")               # remove the in-pipeline encoder
    return Pipeline([("pre", ColumnTransformer([("base", pre, RAW_COLS),
                                                ("leak", "passthrough", ["Suburb_te"])])),
                     ("model", Ridge(alpha=1.0))])

def experiment(yv):
    S = X[["Suburb"]].astype(str)
    XL = X.copy()
    # THE BUG: fitted on all rows, validation labels included, no cross-fitting
    XL["Suburb_te"] = TargetEncoder(target_type="continuous", smooth="auto").fit(S, yv).transform(S).ravel()
    a_ok   = cross_val_score(build_pipeline(), X, yv, cv=cv, scoring="r2")
    a_leak = cross_val_score(leaky_target_encoding(), XL, yv, cv=cv, scoring="r2")

    # MILD BUG: the whole label-free preprocessing fitted on all rows, then CV on the matrix
    b_ok   = cross_val_score(build_pipeline(suburb_encoding="onehot"), X, yv, cv=cv, scoring="r2")
    Z = build_preprocessor("onehot").fit(X, yv).transform(X)
    b_leak = cross_val_score(Ridge(alpha=1.0), Z, yv, cv=cv, scoring="r2")
    return a_ok, a_leak, b_ok, b_leak

rows = []
for label, yv in [("real log(Price)", y),
                  ("shuffled target (pure noise)", pd.Series(np.random.default_rng(0).permutation(y.values)))]:
    a_ok, a_leak, b_ok, b_leak = experiment(yv)
    for name, ok, leak in [("A: Suburb target-encoded outside", a_ok, a_leak),
                           ("B: scaler/imputer/clipper fitted outside", b_ok, b_leak)]:
        d = leak - ok
        rows.append({"target": label, "break": name, "correct R2": ok.mean(), "leaky R2": leak.mean(),
                     "difference": d.mean(), "folds where leaky > correct": f"{(d > 0).sum()}/5",
                     "direction": "inflated" if d.mean() > 0 else "deflated"})
res = pd.DataFrame(rows).round(4)
pd.set_option("display.width", 250); display(res)

ModuleNotFoundError: No module named 'pipeline'

In [4]:
import os
print("Current Working Directory:", os.getcwd())
print("Files in this directory:", os.listdir('.'))

Current Working Directory: /content
Files in this directory: ['.config', 'pipeline.py', 'sample_data']


In [ ]:
n = int(len(X) * 0.8)
Xtr, Xte, ytr = X.iloc[:n], X.iloc[n:], y.iloc[:n]
m = build_pipeline().fit(Xtr, ytr)

row = Xte.iloc[[0]].copy()
for c, v in dict(Suburb="Atlantis", Type="z", Method="??", SellerG="NewAgent",
                 CouncilArea="Nowhere").items():
    row[c] = v
print("absent from training:", {c: row[c].iloc[0] not in set(Xtr[c].dropna())
                                for c in ["Suburb", "Type", "Method", "SellerG", "CouncilArea"]})

# 1) correct pipeline: handle_unknown="ignore" / "infrequent_if_exist" / target-mean fallback
print("correct pipeline ->", m.predict(row).round(3))

# 2) what happens without handle_unknown="ignore" on the low-cardinality encoder
strict = build_pipeline()
strict.set_params(pre__encode_scale__low__handle_unknown="error")
strict.fit(Xtr, ytr)
try:
    strict.predict(row)
except ValueError as e:
    print("without ignore ->", type(e).__name__, ":", str(e)[:150])

# 3) what the unseen Suburb becomes: the global training mean
ct = m.named_steps["pre"].named_steps["encode_scale"]
cleaned = m.named_steps["pre"][:3].transform(row)
print("unseen Suburb encoded as:", ct.named_transformers_["suburb"].transform(cleaned[["Suburb"]]).ravel().round(4),
      "| training mean of log price:", round(ytr.mean(), 4))

In [3]:
%%writefile pipeline.py
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge

RAW_COLS = [
    'Rooms', 'Type', 'Method', 'Distance', 'Bedroom2',
    'Bathroom', 'Car', 'Landsize', 'BuildingArea',
    'YearBuilt', 'CouncilArea', 'Suburb'
]

def load_data(filepath):
    df = pd.read_csv(filepath)
    df = df.dropna(subset=['Price'])
    y = np.log(df['Price'])
    X = df.drop(columns=['Price'], errors='ignore')
    return X, y

def build_preprocessor(suburb_encoding="target"):
    numeric_cols = [
        'Rooms', 'Distance', 'Bedroom2', 'Bathroom',
        'Car', 'Landsize', 'BuildingArea', 'YearBuilt'
    ]
    if suburb_encoding == "target":
        suburb_transformer = TargetEncoder(target_type="continuous", smooth="auto")
    else:
        suburb_transformer = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    return ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_cols),
            ('encode_scale', suburb_transformer, ['Suburb'])
        ],
        remainder='drop'
    )

def build_pipeline(suburb_encoding="target"):
    pre = build_preprocessor(suburb_encoding)
    return Pipeline(steps=[
        ('pre', pre),
        ('model', Ridge(alpha=1.0))
    ])

Writing pipeline.py


In [3]:
# n = int(len(X) * 0.8)
# Xtr, Xte, ytr = X.iloc[:n], X.iloc[n:], y.iloc[:n]
# m = build_pipeline().fit(Xtr, ytr)

# row = Xte.iloc[[0]].copy()
# for c, v in dict(Suburb="Atlantis", Type="z", Method="??", SellerG="NewAgent",
#                  CouncilArea="Nowhere").items():
#     row[c] = v
# print("absent from training:", {c: row[c].iloc[0] not in set(Xtr[c].dropna())
#                                 for c in ["Suburb", "Type", "Method", "SellerG", "CouncilArea"]})

# # 1) correct pipeline: handle_unknown="ignore" / "infrequent_if_exist" / target-mean fallback
# print("correct pipeline ->", m.predict(row).round(3))

# # 2) what happens without handle_unknown="ignore" on the low-cardinality encoder
# strict = build_pipeline()
# strict.set_params(pre__encode_scale__low__handle_unknown="error")
# strict.fit(Xtr, ytr)
# try:
#     strict.predict(row)
# except ValueError as e:
#     print("without ignore ->", type(e).__name__, ":", str(e)[:150])

# # 3) what the unseen Suburb becomes: the global training mean
# ct = m.named_steps["pre"].named_steps["encode_scale"]
# cleaned = m.named_steps["pre"][:3].transform(row)
# print("unseen Suburb encoded as:", ct.named_transformers_["suburb"].transform(cleaned[["Suburb"]]).ravel().round(4),
#       "| training mean of log price:", round(ytr.mean(), 4))

In [2]:
# import os
# import pandas as pd
# import kagglehub
# from pipeline import load_data, build_pipeline

# # 1. Download the Melbourne Housing dataset via kagglehub
# path = kagglehub.dataset_download("anthonypino/melbourne-housing-market")
# print("Dataset files:", os.listdir(path))

# # 2. Load X and y using your pipeline function
# X, y = load_data(os.path.join(path, "Melbourne_housing_FULL.csv"))

# # 3. Run your split, fit, and testing code
# n = int(len(X) * 0.8)
# Xtr, Xte, ytr = X.iloc[:n], X.iloc[n:], y.iloc[:n]
# m = build_pipeline().fit(Xtr, ytr)

# row = Xte.iloc[[0]].copy()
# for c, v in dict(Suburb="Atlantis", Type="z", Method="??", SellerG="NewAgent",
#                  CouncilArea="Nowhere").items():
#     row[c] = v
# print("absent from training:", {c: row[c].iloc[0] not in set(Xtr[c].dropna())
#                                 for c in ["Suburb", "Type", "Method", "SellerG", "CouncilArea"]})

# # 1) correct pipeline prediction
# print("correct pipeline ->", m.predict(row).round(3))

# # 2) what happens without handle_unknown="ignore" on the low-cardinality encoder
# strict = build_pipeline()
# strict.set_params(pre__encode_scale__low__handle_unknown="error")
# strict.fit(Xtr, ytr)
# try:
#     strict.predict(row)
# except ValueError as e:
#     print("without ignore ->", type(e).__name__, ":", str(e)[:150])

# # 3) what the unseen Suburb becomes: the global training mean
# ct = m.named_steps["pre"].named_steps["encode_scale"]
# cleaned = m.named_steps["pre"][:3].transform(row)
# print("unseen Suburb encoded as:", ct.named_transformers_["suburb"].transform(cleaned[["Suburb"]]).ravel().round(4),
#       "| training mean of log price:", round(ytr.mean(), 4))

In [1]:
# import os
# import kagglehub
# import numpy as np
# import pandas as pd
# from sklearn.model_selection import KFold, cross_val_score
# from sklearn.pipeline import Pipeline
# from sklearn.preprocessing import OneHotEncoder
# from pipeline import load_data, build_pipeline, build_preprocessor, RAW_COLS

# # 1. Download/locate dataset via kagglehub
# path = kagglehub.dataset_download("anthonypino/melbourne-housing-market")
# csv_path = os.path.join(path, "Melbourne_housing_FULL.csv")

# # 2. Load data using the correct full path
# X, y = load_data(csv_path)
# cv = KFold(n_splits=5, shuffle=True, random_state=42)

# print("="*60)
# print("EXPERIMENT 1: THE DELIBERATE LEAK TEST")
# print("="*60)

# def leaky_pipeline_experiment():
#     """Builds a pipeline where preprocessing/scaling is fitted globally on all of X, y first."""
#     # Correct baseline
#     correct_scores = cross_val_score(build_pipeline(), X, y, cv=cv, scoring="r2")

#     # Leaky version: Fit the preprocessor on the full dataset beforehand (leakage)
#     leaky_pre = build_preprocessor("target")
#     leaky_pre.fit(X, y)

#     # Wrap pre-fitted preprocessor with Ridge model evaluated via cross_val_score
#     leaky_model = Pipeline([
#         ("pre_fitted", leaky_pre),
#         ("model", build_pipeline().named_steps["model"])
#     ])

#     leaky_scores = cross_val_score(leaky_model, X, y, cv=cv, scoring="r2")

#     diff = leaky_scores - correct_scores
#     print(f"Correct R2 Mean:   {correct_scores.mean():.4f}")
#     print(f"Leaky R2 Mean:     {leaky_scores.mean():.4f}")
#     print(f"Mean Difference:   {diff.mean():.4f} ({'inflated' if diff.mean() > 0 else 'deflated'})")
#     print(f"Folds where leaky > correct: {(diff > 0).sum()}/5")

# leaky_pipeline_experiment()

# print("\n" + "="*60)
# print("EXPERIMENT 2: THE UNSEEN CATEGORY TEST")
# print("="*60)

# # Split train/test to simulate production unseen data
# n = int(len(X) * 0.8)
# Xtr, Xte, ytr = X.iloc[:n], X.iloc[n:], y.iloc[:n]

# # Fit correct pipeline
# m = build_pipeline().fit(Xtr, ytr)

# # Construct a test row with completely fake/unseen categories
# row = Xte.iloc[[0]].copy()
# for c, v in dict(Suburb="AtlantisCity", Type="MegaMansions", Method="FakeMethod",
#                  CouncilArea="OuterSpace", SellerG="GhostAgent").items():
#     if c in row.columns:
#         row[c] = v

# print("Test row injected with unseen categories successfully.")

# # 1) Correct pipeline handles it gracefully
# try:
#     pred = m.predict(row)
#     print(f"-> Correct pipeline prediction for unseen categories: {pred[0]:.3f} (Success: No crash)")
# except Exception as e:
#     print(f"-> Correct pipeline failed: {e}")

# # 2) What happens if a categorical encoder inside lacks unknown handling
# try:
#     # Force an error by creating a strict encoder configuration
#     strict_pre = build_preprocessor("target")
#     # Swap low cardinality encoder to raise error on unknown categories
#     strict_pre.named_steps["encode_scale"].transformers[4] = (
#         "low_strict", OneHotEncoder(handle_unknown="error"), ["Type"]
#     )
#     strict_model = Pipeline([("pre", strict_pre), ("model", build_pipeline().named_steps["model"])])
#     strict_model.fit(Xtr, ytr)
#     strict_model.predict(row)
# except ValueError as e:
#     print(f"-> Without unknown handling (strict mode) -> Raised Expected Error:\n   {type(e).__name__}: {str(e)[:120]}...")

In [11]:
import os
import kagglehub
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from pipeline import load_data, build_pipeline, build_preprocessor, RAW_COLS

# Download and load the Melbourne Housing dataset
path = kagglehub.dataset_download("anthonypino/melbourne-housing-market")
csv_path = os.path.join(path, "Melbourne_housing_FULL.csv")

X, y = load_data(csv_path)
cv = KFold(n_splits=5, shuffle=True, random_state=42)
print("Data loaded successfully. X shape:", X.shape, "| y shape:", y.shape)

Using Colab cache for faster access to the 'melbourne-housing-market' dataset.
Data loaded successfully. X shape: (27247, 20) | y shape: (27247,)


In [12]:
print("="*60)
print("EXPERIMENT 1: THE DELIBERATE LEAK TEST")
print("="*60)

def leaky_pipeline_experiment():
    # 1. Correct baseline: Pipeline learns inside cross_val_score folds
    correct_scores = cross_val_score(build_pipeline(), X, y, cv=cv, scoring="r2")

    # 2. Leaky version: Preprocessor fitted globally on the entire dataset first
    leaky_pre = build_preprocessor("target")
    leaky_pre.fit(X, y)

    # Wrap pre-fitted preprocessor with the model estimator
    leaky_model = Pipeline([
        ("pre_fitted", leaky_pre),
        ("model", build_pipeline().named_steps["model"])
    ])

    leaky_scores = cross_val_score(leaky_model, X, y, cv=cv, scoring="r2")

    # Report findings
    diff = leaky_scores - correct_scores
    print(f"Correct R2 Mean:             {correct_scores.mean():.4f}")
    print(f"Leaky R2 Mean:               {leaky_scores.mean():.4f}")
    print(f"Mean Difference:             {diff.mean():.4f} ({'inflated' if diff.mean() > 0 else 'deflated'})")
    print(f"Folds where leaky > correct: {(diff > 0).sum()}/5")

leaky_pipeline_experiment()

EXPERIMENT 1: THE DELIBERATE LEAK TEST
Correct R2 Mean:             0.6325
Leaky R2 Mean:               0.6320
Mean Difference:             -0.0005 (deflated)
Folds where leaky > correct: 2/5


Experiment 2 — The Unseen Category Test

In [16]:
print("="*60)
print("EXPERIMENT 2: THE UNSEEN CATEGORY TEST")
print("="*60)

# Split train/test to simulate production data
n = int(len(X) * 0.8)
Xtr, Xte, ytr = X.iloc[:n], X.iloc[n:], y.iloc[:n]

# Fit the correct pipeline on training data
m = build_pipeline().fit(Xtr, ytr)

# Construct a test row containing entirely fake/unseen categories
row = Xte.iloc[[0]].copy()
for c, v in dict(Suburb="AtlantisCity", Type="MegaMansions", Method="FakeMethod",
                 CouncilArea="OuterSpace", SellerG="GhostAgent").items():
    if c in row.columns:
        row[c] = v

print("Test row injected with unseen categories successfully.")

# 1) Correct pipeline execution (handles unseen categories smoothly)
try:
    pred = m.predict(row)
    print(f"-> Correct pipeline prediction for unseen categories: {pred[0]:.3f} (Success: No crash)")
except Exception as e:
    print(f"-> Correct pipeline failed: {e}")

# 2) Strict mode execution (forcing an error via a strict OneHotEncoder)
try:
    from sklearn.compose import ColumnTransformer
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder

    # Create a strict pipeline that explicitly uses handle_unknown="error" for categorical columns
    strict_model = Pipeline([
        ("pre", ColumnTransformer([
            ("low_strict", OneHotEncoder(handle_unknown="error"), ["Type"])
        ], remainder="drop")),
        ("model", m.named_steps["model"])
    ])

    strict_model.fit(Xtr, ytr)
    strict_model.predict(row)
except ValueError as e:
    print(f"-> Without unknown handling (strict mode) -> Raised Expected Error:\n   {type(e).__name__}: {str(e)[:150]}...")

EXPERIMENT 2: THE UNSEEN CATEGORY TEST
Test row injected with unseen categories successfully.
-> Correct pipeline prediction for unseen categories: 14.038 (Success: No crash)
-> Without unknown handling (strict mode) -> Raised Expected Error:
   ValueError: Found unknown categories ['MegaMansions'] in column 0 during transform...
